## 2. Service Health Check

In [1]:
import sys
import os
from pathlib import Path
import requests
import time

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

# Find project root
current_dir = Path.cwd()
if current_dir.name == "week7" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent.parent

if project_root.exists():
    print(f"Project root: {project_root}")
    sys.path.insert(0, str(project_root))
else:
    print("⚠ Project root not found - check directory structure")

# Load .env file if it exists
env_file = project_root / ".env"
if env_file.exists():
    print(f"\n✓ Loading environment from: {env_file}")
    with open(env_file) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, value = line.split('=', 1)
                if key not in os.environ:
                    os.environ[key] = value
    print("✓ Environment variables loaded")
else:
    print(f"\n⚠ No .env file found at: {env_file}")
    print("  Run: cp .env.example .env")
    print("  Then add your JINA_API_KEY, LANGFUSE_PUBLIC_KEY, and LANGFUSE_SECRET_KEY")

# Configuration for notebook tests
REQUEST_TIMEOUT = 300
TRUNCATE_ANSWERS = True
TRUNCATE_LENGTH = 200

# OpenRouter: set OPENROUTER_API_KEY in project .env (server-side); model id below is the default slug.
OPENROUTER_DEFAULT_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o-mini")
OPENROUTER_TRANSLATION_MODEL = os.environ.get("OPENROUTER_TRANSLATION_MODEL", os.environ.get("TRANSLATION_MODEL", ""))

print("\n✓ Setup complete")
print(f"  Default OpenRouter model (OPENROUTER_MODEL): {OPENROUTER_DEFAULT_MODEL}")
if OPENROUTER_TRANSLATION_MODEL.strip():
    print(f"  Translation model (OPENROUTER_TRANSLATION_MODEL or TRANSLATION_MODEL): {OPENROUTER_TRANSLATION_MODEL}")
else:
    print("  Translation model: (unset — same as default)")

Python Version: 3.12.12
Project root: /Users/anhvietpham/Documents/AI/Project-practice/production-agentic-rag

✓ Loading environment from: /Users/anhvietpham/Documents/AI/Project-practice/production-agentic-rag/.env
✓ Environment variables loaded

✓ Setup complete
  Default OpenRouter model (OPENROUTER_MODEL): openai/gpt-4o-mini
  Translation model (OPENROUTER_TRANSLATION_MODEL or TRANSLATION_MODEL): google/gemini-2.0-flash-001


In [2]:
print("WEEK 7 SERVICE HEALTH CHECK")
print("=" * 40)

API_HEALTH = "http://localhost:8001/api/v1/health"
all_healthy = True

try:
    response = requests.get(API_HEALTH, timeout=10)
    if response.status_code == 200:
        body = response.json()
        print(f"✓ FastAPI: {body.get('status', 'ok')}")
        services = body.get("services") or {}
        llm = services.get("llm") or services.get("ollama")
        if llm:
            print(f"✓ LLM: {llm.get('message', llm)}")
        prov = (llm or {}).get("provider", "")
        if prov == "openrouter" or "openrouter" in (llm or {}).get("message", "").lower():
            print(f"  → Server uses OpenRouter ({OPENROUTER_DEFAULT_MODEL}) for /ask and LangGraph.")
        elif (llm or {}).get("status") == "unhealthy":
            print("  ⚠ LLM unhealthy — set OPENROUTER_API_KEY in the API .env and restart.")
            all_healthy = False
        else:
            print("  ⚠ Unexpected LLM health payload; check API logs.")
    else:
        print(f"✗ FastAPI health HTTP {response.status_code}")
        all_healthy = False
except Exception as e:
    print(f"✗ FastAPI not accessible: {e}")
    all_healthy = False

if all_healthy:
    print("\n✓ Ready for Week 7 (LLM via OpenRouter when OPENROUTER_API_KEY is set on the server).")
else:
    print("\n⚠ Fix API / LLM before running RAG cells.")

WEEK 7 SERVICE HEALTH CHECK
✓ FastAPI: ok
✓ LLM: openrouter: OpenRouter API reachable — model=openai/gpt-4o-mini, translation_model=google/gemini-2.0-flash-001
  → Server uses OpenRouter (openai/gpt-4o-mini) for /ask and LangGraph.

✓ Ready for Week 7 (LLM via OpenRouter when OPENROUTER_API_KEY is set on the server).


## 3. Test Traditional RAG

In [3]:
print("TRADITIONAL RAG TEST (Baseline)")
print("=" * 40)
print(
    "NOTE: Generation runs on the FastAPI process. Put OPENROUTER_API_KEY in the project\n"
    "      .env next to compose.yml, then restart: uv run -m src.main\n"
    "      (Loading .env in this notebook only affects the kernel, not the API.)\n"
)

# Preflight: server should report OpenRouter when the key is set
try:
    h = requests.get("http://localhost:8001/api/v1/health", timeout=10).json()
    llm = (h.get("services") or {}).get("llm") or (h.get("services") or {}).get("ollama")
    if llm:
        msg = (llm.get("message") or "").lower()
        if llm.get("provider") == "openrouter" or "openrouter" in msg:
            print("✓ API LLM: OpenRouter (Traditional RAG uses OPENROUTER_MODEL from server .env)\n")
        else:
            print("⚠ API LLM may be misconfigured — ensure OPENROUTER_API_KEY is set on the API process.")
            print(f"   Health message: {llm.get('message')}\n")
except Exception as e:
    print(f"⚠ Could not read /health before RAG test: {e}\n")

question = "What are LLM Self-Refinement?"
print(f"Question: {question}\n")

start_time = time.time()

try:
    response = requests.post(
        "http://localhost:8001/api/v1/ask",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
            "model": OPENROUTER_DEFAULT_MODEL,
        },
        timeout=REQUEST_TIMEOUT
    )
    
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Traditional RAG ({elapsed:.1f}s)")
        
        # Display answer with configurable truncation
        answer = data['answer']
        if TRUNCATE_ANSWERS and len(answer) > TRUNCATE_LENGTH:
            print(f"\nAnswer: {answer[:TRUNCATE_LENGTH]}...")
            print(f"(truncated, full length: {len(answer)} chars)")
        else:
            print(f"\nAnswer: {answer}")
        
        # Display sources with validation
        sources = data.get('sources', [])
        print(f"\nSources: {len(sources)} papers")
        if sources:
            for i, source in enumerate(sources[:3], 1):  # Show first 3
                if isinstance(source, dict):
                    print(f"  {i}. {source.get('title', 'Unknown')}")
                else:
                    print(f"  {i}. {source}")
        
        print(f"Search mode: {data.get('search_mode', 'unknown')}")
    else:
        print(f"✗ Request failed: {response.status_code}")
        detail = response.text[:800]
        print(detail)
        if "openrouter" in detail.lower() or "api key" in detail.lower():
            print(
                "\n→ Fix: add OPENROUTER_API_KEY to .env in the API project root, restart uvicorn,\n"
                "   then confirm startup log shows: LLM provider=openrouter ..."
            )
        
except Exception as e:
    print(f"✗ Error: {e}")

TRADITIONAL RAG TEST (Baseline)
NOTE: Generation runs on the FastAPI process. Put OPENROUTER_API_KEY in the project
      .env next to compose.yml, then restart: uv run -m src.main
      (Loading .env in this notebook only affects the kernel, not the API.)

✓ API LLM: OpenRouter (Traditional RAG uses OPENROUTER_MODEL from server .env)

Question: What are LLM Self-Refinement?

✓ Traditional RAG (0.0s)

Answer: LLM Self-Refinement refers to a process within the De Jure framework, which is described as an automated pipeline designed for extracting structured regulatory rules from documents. This process invol...
(truncated, full length: 1192 chars)

Sources: 1 papers
  1. https://arxiv.org/pdf/2604.02276.pdf
Search mode: hybrid


## 4 Test Agentic RAG - Scenario 1: Out-of-scope Rejection

In [4]:
print("AGENTIC RAG - SCENARIO 1: Out-of-scope Rejection")
print("=" * 50)

question = "What is a dog?"
print(f"Question: {question}")
print("Expected: Guardrail should reject (score < 60) and explain scope\n")

start_time = time.time()

try: 
    response = requests.post(
        "http://localhost:8001/api/v1/ask-agentic",
        json={
            "query": question, 
            "top_k": 3,
            "use_hybrid": True,
        },
        timeout=REQUEST_TIMEOUT
    )

    elapsed = time.time() - start_time

    if response.status_code == 200:
        data = response.json()
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")
        print(f"\nAnswer: {data['answer']}")
        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1): 
            print(f"Step {i}:")

        # Check if guardrail score is in reasing steps
        guardrail_step = next(
            (s for s in data.get('reasoning_steps', []) if 'validated' in s.lower() and 'score' in s.lower()),
            None
        )
        if guardrail_step:
            print(f"\nGuardrail validation: {guardrail_step}")
        
        if data.get('retrieval_attempts', 0) == 0:
            print("\n✓ SUCCESS: Query correctly rejected by guardrail (no retrieval)!")
        else:
            print("\n⚠ UNEXPECTED: Query should have been rejected without retrieval")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Error: {e}")

AGENTIC RAG - SCENARIO 1: Out-of-scope Rejection
Question: What is a dog?
Expected: Guardrail should reject (score < 60) and explain scope

✓ Agentic RAG (1.7s)

Answer: I apologize, but I can only help with questions about academic research papers in Computer Science, Artificial Intelligence, and Machine Learning from arXiv.

Your question: 'What is a dog?'

This appears to be outside my domain of expertise. For questions like this, you might want to try:
- General-purpose AI assistants for broad knowledge questions
- Domain-specific resources for topics outside CS/AI/ML
- Technical documentation if asking about specific software/tools

If you have a question about AI/ML research papers, I'd be happy to help!

Retrieval attempts: 0

Reasoning steps:
Step 1:
Step 2:

Guardrail validation: Validated query scope (score: 0/100)

✓ SUCCESS: Query correctly rejected by guardrail (no retrieval)!


## 5. Test Agentic RAG - Scenario 2: Successful Retrieval

In [5]:
print("AGENTIC RAG - SCENARIO 2: Successful Retrieval")
print("=" * 50)

question = "Summarize how iterative self-refinement is used in the De Jure paper for structured extraction."
print(f"Question: {question}")
print("Expected: Guardrail should pass (score >= 60) and retrieve relevant papers\n")

start_time = time.time()

try:
    response = requests.post(
        "http://localhost:8001/api/v1/ask-agentic",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
        },
        timeout=REQUEST_TIMEOUT
    )
    
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")
        
        # Display answer with better formatting
        answer = data.get('answer', '')
        print(f"\nAnswer:\n{'-'*50}")
        if TRUNCATE_ANSWERS and len(answer) > 500:  # Use longer limit for detailed answers
            print(answer[:500] + "...")
            print(f"(truncated, full length: {len(answer)} chars)")
        else:
            print(answer)
        print('-'*50)
        
        # Display sources with validation
        sources = data.get('sources', [])
        print(f"\nSources: {len(sources)} papers")
        if sources:
            for i, source in enumerate(sources, 1):
                if isinstance(source, dict):
                    print(f"  {i}. {source.get('title', source.get('id', 'Unknown'))}")
                elif isinstance(source, str):
                    print(f"  {i}. {source}")
                else:
                    print(f"  {i}. {str(source)}")
        
        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1):
            print(f"  {i}. {step}")
        

        # Check rewritten_query field
        if data.get('rewritten_query') is None:
            print("\n✓ Query was not rewritten (worked on first attempt)")
        else:
            print(f"\n→ Query was rewritten to: {data['rewritten_query']}")
        
        if data.get('retrieval_attempts', 0) >= 1:
            print("\n✓ SUCCESS: Agent retrieved and used documents!")
        else:
            print("\n⚠ UNEXPECTED: Agent didn't retrieve for research question")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Error: {e}")


AGENTIC RAG - SCENARIO 2: Successful Retrieval
Question: Summarize how iterative self-refinement is used in the De Jure paper for structured extraction.
Expected: Guardrail should pass (score >= 60) and retrieve relevant papers

✓ Agentic RAG (7.2s)

Answer:
--------------------------------------------------
The De Jure paper presents an iterative self-refinement process as a core component of its automated pipeline for extracting structured regulatory rules from documents. This process is designed to enhance the quality and accuracy of the extracted rules through a systematic approach that includes multiple stages of generation, judgment, and selective repair.

1. **Decoupling Extraction from Verification**: The De Jure pipeline separates the generation of rules from their evaluation. This allows t...
(truncated, full length: 2129 chars)
--------------------------------------------------

Sources: 1 papers
  1. ## De Jure: Iterative LLM Self-Refinement for Structured Extraction of Reg

## 6 Test Agentic RAG - Scenario 3: Query Rewriting

In [6]:
print("AGENTIC RAG - SCENARIO 3: Query Rewriting")
print("=" * 50)

question = "what is Iterative LLM?"
print(f"Question: {question}")
print("Expected: Agent may rewrite query if documents aren't relevant\n")

start_time = time.time()

try: 
    response = requests.post(
        "http://localhost:8001/api/v1/ask-agentic",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
        },
        timeout=REQUEST_TIMEOUT
    )

    elapsed = time.time() - start_time

    if response.status_code == 200: 
        data = response.json()
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")

        # Display answer with better formatting
        answer = data.get('answer', '')
        print(f"\nAnswer:\n{'-'*50}")
        if TRUNCATE_ANSWERS and len(answer) > 500: 
            print(answer[:500] + "...")
            print(f"(truncated, full length: {len(answer)} chars)")
        else: 
            print(answer)
        print('-'*50)

        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1):
            print(f"  {i}. {step}")
        
        # Check for guardrail validation step
        print("\nValidating guardrail and rewrite steps:")
        reasoning_steps = data.get('reasoning_steps', [])
        if any("validated" in step.lower() for step in reasoning_steps):
            guardrail_step = next(s for s in reasoning_steps if "validated" in s.lower())
            print(f"  ✓ Guardrail validation: {guardrail_step}")
        else:
            print("  ⚠ Guardrail validation step missing")

        # Check for query rewriting
        if data.get('rewritten_query'):
            print(f"\n✓ Query was rewritten!")
            print(f"  Original: {question}")
            print(f"  Rewritten: {data['rewritten_query']}")
        elif data.get('retrieval_attempts', 0) > 1:
            print("\n→ Multiple retrieval attempts detected")
            if any("rewritten" in step.lower() for step in reasoning_steps):
                print("  ✓ Rewrite step found in reasoning")
            else:
                print("  ⚠ Multiple attempts but no rewrite info")
        else:
            print("\n→ Query worked on first attempt (no rewrite needed)")
        
        if data.get('retrieval_attempts', 0) > 1:
            print(f"\n✓ Agent performed {data['retrieval_attempts']} retrieval attempts")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Error: {e}")



AGENTIC RAG - SCENARIO 3: Query Rewriting
Question: what is Iterative LLM?
Expected: Agent may rewrite query if documents aren't relevant

✓ Agentic RAG (8.8s)

Answer:
--------------------------------------------------
I apologize, but I couldn't find relevant research papers after 2 attempts.
This may be because:
1. No papers in the database contain relevant information
2. The query terms don't match the indexed content

Please try rephrasing your question with more specific technical terms.
--------------------------------------------------

Retrieval attempts: 2

Reasoning steps:
  1. Validated query scope (score: 80/100)
  2. Retrieved documents (2 attempt(s))
  3. Graded documents (0 relevant)
  4. Rewritten query for better results
  5. Generated answer from retrieved papers

Validating guardrail and rewrite steps:
  ✓ Guardrail validation: Validated query scope (score: 80/100)

→ Multiple retrieval attempts detected
  ✓ Rewrite step found in reasoning

✓ Agent performed 2 retri

In [7]:
print("AGENTIC RAG - SCENARIO 4: Multiple Out-of-Scope Queries")
print("=" * 50)

test_queries = [
    ("What is a dog?", "Biology question"),
    ("What's the weather today?", "Weather question"),
    ("Hello, how are you?", "Greeting"),
]

print("Testing guardrail rejection with various non-ML/NLP queries:\n")

print("Testing guardrail rejection with various non-ML/NLP queries:\n")

for query, description in test_queries:
    print(f"Query: {query}")
    print(f"Type: {description}")
    
    try:
        response = requests.post(
            "http://localhost:8001/api/v1/ask-agentic",
            json={"query": query, "top_k": 3, "use_hybrid": True},
            timeout=30
        )
        
        if response.status_code == 200:
            data = response.json()
            
            # Check if rejected (no retrieval)
            is_rejected = data['retrieval_attempts'] == 0
            
            # Get guardrail score from reasoning if available
            guardrail_step = next(
                (s for s in data['reasoning_steps'] if 'validated' in s.lower() and 'score' in s.lower()),
                None
            )
            
            print(f"Result: {'✓ REJECTED' if is_rejected else '✗ ACCEPTED'} (attempts: {data['retrieval_attempts']})")
            if guardrail_step:
                print(f"Guardrail: {guardrail_step}")
        else:
            print(f"✗ Request failed: {response.status_code}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    print("-" * 50)

AGENTIC RAG - SCENARIO 4: Multiple Out-of-Scope Queries
Testing guardrail rejection with various non-ML/NLP queries:

Testing guardrail rejection with various non-ML/NLP queries:

Query: What is a dog?
Type: Biology question
Result: ✓ REJECTED (attempts: 0)
Guardrail: Validated query scope (score: 0/100)
--------------------------------------------------
Query: What's the weather today?
Type: Weather question
Result: ✓ REJECTED (attempts: 0)
Guardrail: Validated query scope (score: 0/100)
--------------------------------------------------
Query: Hello, how are you?
Type: Greeting
Result: ✓ REJECTED (attempts: 0)
Guardrail: Validated query scope (score: 0/100)
--------------------------------------------------


# 8 Interactive Testing 

In [8]:
def ask_agentic(question: str, show_full_answer: bool = False):
    """Helper function to test agentic RAG.
    
    Args:
        question: The question to ask
        show_full_answer: If True, show full answer regardless of TRUNCATE_ANSWERS setting
    """
    print(f"Question: {question}\n")
    
    start = time.time()
    
    try:
        response = requests.post(
            "http://localhost:8001/api/v1/ask-agentic",
            json={"query": question, "top_k": 3, "use_hybrid": True},
            timeout=REQUEST_TIMEOUT
        )
        
        elapsed = time.time() - start
        
        if response.status_code == 200:
            data = response.json()
            print(f"✓ Response in {elapsed:.1f}s\n")
            
            # Display answer
            answer = data.get('answer', '')
            print(f"Answer:\n{'-'*50}")
            if not show_full_answer and TRUNCATE_ANSWERS and len(answer) > 500:
                print(answer[:500] + "...")
                print(f"(truncated, full length: {len(answer)} chars)")
            else:
                print(answer)
            print('-'*50)
            
            # Display metadata
            print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
            
            # Display sources with validation
            sources = data.get('sources', [])
            print(f"Sources: {len(sources)}")
            if sources:
                for i, source in enumerate(sources[:3], 1):  # Show first 3
                    if isinstance(source, dict):
                        print(f"  {i}. {source.get('title', source.get('id', 'Unknown'))}")
                    elif isinstance(source, str):
                        print(f"  {i}. {source}")
            
            # Display reasoning
            print(f"\nReasoning:")
            for step in data.get('reasoning_steps', []):
                print(f"  • {step}")
        else:
            print(f"✗ Error: {response.status_code}")
            print(response.text)
    except Exception as e:
        print(f"✗ Exception: {e}")

# Try it!
ask_agentic("How does BERT differ from GPT?")

Question: How does BERT differ from GPT?

✓ Response in 9.8s

Answer:
--------------------------------------------------
I apologize, but I couldn't find relevant research papers after 2 attempts.
This may be because:
1. No papers in the database contain relevant information
2. The query terms don't match the indexed content

Please try rephrasing your question with more specific technical terms.
--------------------------------------------------

Retrieval attempts: 2
Sources: 0

Reasoning:
  • Validated query scope (score: 90/100)
  • Retrieved documents (2 attempt(s))
  • Graded documents (0 relevant)
  • Rewritten query for better results
  • Generated answer from retrieved papers
